<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
import os

if not os.path.exists('FlyRank_AI'):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git

os.chdir('FlyRank_AI')

!python scripts/01_prepare_features.py

import sys
sys.path.append('scripts')

import json
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(
    'data/processed/refresh_feature_vector.csv'
)

print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/FlyRank_AI/FlyRank_AI/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


In [23]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = order[:k]
    return np.asarray(y_true)[top_k].mean()

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [24]:
from sklearn.model_selection import GroupKFold, cross_val_predict

feature_cols = [
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update'
]

model_data = df.dropna(
    subset=feature_cols + [
        'is_declining_label',
        'client_id'
    ]
).copy()

X = model_data[feature_cols]
y = model_data['is_declining_label']
groups = model_data['client_id']

# Confirm the target is binary
target_values = sorted(y.dropna().unique())

assert target_values == [0, 1], (
    f'Expected binary target [0, 1], found {target_values}'
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

gkf = GroupKFold(n_splits=5)

# Generate out-of-fold predictions
oof_probs = cross_val_predict(
    rf,
    X,
    y,
    groups=groups,
    cv=gkf,
    method='predict_proba',
    n_jobs=-1
)

# IMPORTANT: create risk_score BEFORE using it
model_data['risk_score'] = oof_probs[:, 1]

print("OOF rows:", len(model_data))
print("OOF clients:", model_data['client_id'].nunique())

print("\nOOF risk score summary:")
print(model_data['risk_score'].describe())

print("\nOOF Precision@K:")
for k in [20, 50, 100, 200]:
    p = precision_at_k(
        y.values,
        model_data['risk_score'].values,
        k
    )
    print(f"P@{k}: {p:.3f}")

median_pos = model_data.loc[
    model_data['avg_position'] > 0,
    'avg_position'
].median()

def reason_code(row):
    if (
        row['days_since_last_update'] >= 180
        and row['impressions_prev_30d'] >= 500
    ):
        return 'stale_but_visible'

    if row['avg_position'] > median_pos:
        return 'weak_position'

    return 'standard_review'

model_data['reason_code'] = model_data.apply(
    reason_code,
    axis=1
)

queue = model_data.sort_values(
    'risk_score',
    ascending=False
).copy()

base_rate = y.mean()

print(
    f'\nBase rate: {base_rate:.3f} ({base_rate:.1%})'
)

queue[
    [
        'content_id',
        'risk_score',
        'reason_code',
        'impressions_prev_30d',
        'avg_position',
        'days_since_last_update'
    ]
].head(20)

OOF rows: 30000
OOF clients: 32

OOF risk score summary:
count    30000.000000
mean         0.554038
std          0.305999
min          0.000000
25%          0.330000
50%          0.612583
75%          0.815000
max          1.000000
Name: risk_score, dtype: float64

OOF Precision@K:
P@20: 0.750
P@50: 0.720
P@100: 0.730
P@200: 0.730

Base rate: 0.542 (54.2%)


,content_id,risk_score,reason_code,impressions_prev_30d,avg_position,days_since_last_update
21079,content_751e38250ff3,1.0,standard_review,582,6.7,20
18264,content_ab17bac1b626,1.0,weak_position,9887,37.9,20
17993,content_495e95c93181,1.0,standard_review,83,8.3,20
28578,content_ea7632e60529,1.0,standard_review,31,11.2,104
18111,content_243976103493,1.0,standard_review,5,6.4,20
17839,content_b3a1d8ac4964,1.0,standard_review,4,7.4,8
15812,content_7a9e31119f1e,1.0,standard_review,59,5.9,104
25439,content_3d8f6737ad9d,1.0,standard_review,25,8.6,8
25672,content_01607d2cc325,1.0,weak_position,7077,15.7,20
6502,content_e7c8ebd61165,1.0,standard_review,5,6.4,20


**Ranked actions**:
 Each content item receives an out-of-fold Random Forest risk score and a plain-language reason code. The queue ranks items from highest to lowest predicted decline risk, while The reason code provides a simple review cue for why the item may deserve attention; it is not a causal explanation of the model score.. stale_but_visible means the page has meaningful recent impressions but has not been updated for at least 180 days;

`weak_position` means its average position is worse than the observed median; `standard_review` means the item does not meet either of those stronger conditions.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [25]:
n_clients = groups.nunique()

print(
    f"Queue covers {len(queue):,} content items from {n_clients} clients. "
    f"Risk scores are out-of-fold predictions from {gkf.get_n_splits()}-fold "
    f"grouped cross-validation (GroupKFold on client_id) -- every score reflects "
    f"a model that never saw that content item's client during training."
)

Queue covers 30,000 content items from 32 clients. Risk scores are out-of-fold predictions from 5-fold grouped cross-validation (GroupKFold on client_id) -- every score reflects a model that never saw that content item's client during training.


**Intended use:** A content strategist can use the queue as a starting point for weekly review: pages with higher model scores are reviewed first, rather than automatically changed.

**Limits**: The queue is decision-support, not a prediction of what will happen to an individual page. The scores are pooled out-of-fold predictions from 5-fold client-grouped validation across 32 clients, so they are more conservative than in-sample predictions. The observed base rate is 54.2%, while pooled OOF Precision@20 is 75.0%, Precision@50 is 72.0%, and Precision@100 is 73.0%. These results show directional ranking signal in this dataset, but they do not establish performance on a genuinely new client or future time period.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [26]:
visible = (
    model_data['impressions_prev_30d'] >= 500
).astype(int)

stale = (
    model_data['days_since_last_update'] >= 180
).astype(int)

median_pos = model_data.loc[
    model_data['avg_position'] > 0,
    'avg_position'
].median()

weak_position = (
    model_data['avg_position'] > median_pos
).astype(int)

baseline_flag = (
    visible & (stale | weak_position)
).astype(int)


comparison_threshold = 0.5

model_flag = (
    model_data['risk_score'] >= comparison_threshold
).astype(int)

disagreement = model_data[
    baseline_flag != model_flag
]

print(
    f'{len(disagreement):,} rows where model and baseline '
    f'rule disagree at risk_score >= {comparison_threshold:.1f} '
    f'-- review these first'
)

disagreement[
    [
        'content_id',
        'risk_score',
        'reason_code'
    ]
].head(10)

15,398 rows where model and baseline rule disagree at risk_score >= 0.5 -- review these first


,content_id,risk_score,reason_code
0,content_304f48230142,0.650000,standard_review
3,content_331d6c4de07b,0.515000,standard_review
5,content_d4084a4bc775,0.725000,standard_review
6,content_9a34b442b552,0.797881,standard_review
9,content_c27558df2b0c,0.580000,standard_review
12,content_42fb2cad9ecf,0.585000,standard_review
13,content_a5a2fbc76336,0.615000,weak_position
14,content_91067a14431a,0.730000,weak_position
15,content_689414059706,0.960000,standard_review
16,content_78bd1d4a1d4d,0.565000,standard_review


In [27]:
precision_results = []

for k in [20, 50, 100, 200]:
    precision_results.append({
        'k': k,
        'precision_at_k': precision_at_k(
            y,
            model_data['risk_score'],
            k
        )
    })

precision_results = pd.DataFrame(
    precision_results
)

print('Pooled OOF Precision@K:')
display(precision_results)

Pooled OOF Precision@K:


,k,precision_at_k
0,20,0.75
1,50,0.72
2,100,0.73
3,200,0.73


**What a person must check before acting:**
- For this diagnostic comparison only, a 0.5 probability threshold is used to identify disagreements. The production workflow should rank pages rather than treat 0.5 as a validated decision threshold.
- Any high-ranked row with very low impressions_prev_30d should receive extra scrutiny because sparse traffic can make the observed label and model signal less stable.

**No-go list — never automate:**
- Do not auto-publish or auto-edit content based on this score alone; it flags candidates for
  a human writer/editor, nothing more.
- Do not use this ranking to make client-facing promises about traffic outcomes — no causal
  claim is supported (per writing-honest-claims: cross-sectional data, no intervention design).
- Do not apply this model to a client or content type outside the Refresh lane without
  re-validating — the honest-split numbers here are specific to this data.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signs this playbook has gone stale:**

- If grouped validation performance falls materially across future data refreshes, or if the gap between random and grouped validation widens, re-audit the feature set and client effects.
- If the base rate of `is_declining_label` shifts meaningfully from the measured dataset base rate, the risk scores may need recalibration rather than being reused unchanged.
- If reason codes increasingly fail to match what reviewers find during inspection, the disagreement log should be investigated before continuing to rely on the ranking.
- **Retrain / re-audit trigger:** re-run W06's leakage-and-split audit whenever the raw data is refreshed with a new month of `content_refresh_anonymized.csv`, not only when performance appears to decline. Leakage can return if the feature-preparation script changes.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [28]:
Path('work/outputs').mkdir(
    parents=True,
    exist_ok=True
)

# Action queue: information available when making the ranking.
queue_cols = [
    'content_id',
    'risk_score',
    'reason_code',
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update'
]

queue[queue_cols].to_csv(
    'work/outputs/action_playbook_queue.csv',
    index=False
)

# Evaluation file: includes the observed target for analysis only.
evaluation_cols = queue_cols + [
    'is_declining_label'
]

queue[evaluation_cols].to_csv(
    'work/outputs/action_playbook_evaluation.csv',
    index=False
)

summary = {
    'base_rate': float(base_rate),
    'n_scored': int(len(queue)),
    'n_clients': int(n_clients),
    'cv_n_splits': int(gkf.get_n_splits()),
    'n_disagreements_with_baseline': int(
        len(disagreement)
    ),
    'precision_at_20': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            20
        )
    ),
    'precision_at_50': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            50
        )
    ),
    'precision_at_100': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            100
        )
    ),
    'precision_at_200': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            200
        )
    )
}

with open(
    'work/outputs/playbook_summary.json',
    'w'
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    'Wrote work/outputs/action_playbook_queue.csv, '
    'action_playbook_evaluation.csv and '
    'playbook_summary.json'
)

print(summary)

Wrote work/outputs/action_playbook_queue.csv, action_playbook_evaluation.csv and playbook_summary.json
{'base_rate': 0.5420666666666667, 'n_scored': 30000, 'n_clients': 32, 'cv_n_splits': 5, 'n_disagreements_with_baseline': 15398, 'precision_at_20': 0.75, 'precision_at_50': 0.72, 'precision_at_100': 0.73, 'precision_at_200': 0.73}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.